# AquaTrack: fine-tune RTMW-x on swimmers (body and feet)

RTMW-x gives **133 points**: 17 body, 6 feet (big toe, small toe, heel), 68 face and 21 per hand.
This notebook trains it further on SwimXYZ's computer-generated swimmers, for the **body and feet**.
SwimXYZ has no finger or face labels, so those points are left out of training and keep what RTMW-x already knows.

**Needs:** the 20 shard zips from `aquatrack_swimxyz_full_freestyle.ipynb` in `MyDrive/aquatrack_swim_full/shards/`.
Everything this notebook makes goes in `MyDrive/aquatrack_swim_full/rtmw/`.

| Part | Runtime | What it does | Time (estimate) |
|---|---|---|---|
| 0. Setup | any | Installs MMPose. Run it every session | 3 to 5 min |
| A. Labels | CPU | Adds SwimXYZ's 25-point labels (with feet) to the frames already in your shards. Downloads SwimXYZ's labels zip (about 1.7 GB) once, not the videos | 10 to 30 min |
| B. Train | GPU (A100 or L4 with Colab Pro, else T4) | Fine-tunes RTMW-x. Saves to Drive every epoch and carries on after a disconnect | 30 epochs: roughly 2 to 3 h on a T4, under 1 h on an A100 |
| C. Check and export | any | Scores the stock and swim models on the held-out swimmer, and saves ONNX files for the comparison notebook and the app | 5 to 15 min |

**Tip:** C1 also works before training. It exports the stock RTMW-x so you can try the real RTMW-x on your
clip in Part 4 of `aquatrack_vitpose_vs_yolo.ipynb` straight away.

## 0. Setup (run every session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q "setuptools<81" cython albumentations
!pip install -q --no-build-isolation xtcocotools
!pip install -q mmengine==0.10.7 json_tricks munkres onnx onnxruntime
!pip install -q --no-deps mmcv-lite==2.1.0 mmpose==1.3.2 mmdet==3.3.0

import os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'   # MMPose checkpoints hold more than weights

"""Two small fixes so MMPose 1.3.2 runs on today's Colab. RTMW training doesn't use what they skip.
Safe to run more than once."""
import importlib.util
from pathlib import Path

root = Path(importlib.util.find_spec("mmpose").origin).parent
fixes = [
    # EDPose needs mmcv's compiled ops, which mmcv-lite doesn't have
    (root / "models" / "heads" / "__init__.py",
     "\nfrom .transformer_heads import EDPoseHead\n",
     "\ntry:  # EDPose needs mmcv's compiled ops, which mmcv-lite doesn't have\n"
     "    from .transformer_heads import EDPoseHead\n"
     "except ImportError:\n"
     "    EDPoseHead = None\n"),
    # albumentations 2.x moved its transforms module; this line only decides whether to print a warning
    (root / "datasets" / "transforms" / "common_transforms.py",
     "albumentations.augmentations.transforms, obj_type)",
     "albumentations, obj_type)"),
]
for path, old, new in fixes:
    text = path.read_text()
    if old in text:
        path.write_text(text.replace(old, new))
        print("fixed", path.relative_to(root.parent))

from pathlib import Path
import mmpose
DRIVE = Path('/content/drive/MyDrive/aquatrack_swim_full')
SHARDS = DRIVE / 'shards'                     # made by the YOLO notebook
OUT = DRIVE / 'rtmw'                          # everything from this notebook
ANN = OUT / 'annotations'                     # train.json and val.json (Part A)
RUN = OUT / 'runs' / 'rtmw_x_swim'            # training checkpoints (Part B)
DATASET = Path('/content/swim_dataset')       # local copy of the frames, for speed
RTMW_X = ('https://download.openmmlab.com/mmpose/v1/projects/rtmw/'
          'rtmw-x_simcc-cocktail14_pt-ucoco_270e-384x288-f840f204_20231122.pth')
BASE_CONFIG = (Path(mmpose.__file__).parent / '.mim' / 'configs' / 'wholebody_2d_keypoint' / 'rtmpose'
               / 'cocktail14' / 'rtmw-x_8xb320-270e_cocktail14-384x288.py')
OUT.mkdir(parents=True, exist_ok=True)
print('Shards found:', len(list(SHARDS.glob('*.zip'))), 'of 20')

Helper code used by the cells below (run it every session too).

In [ ]:
%%writefile /content/rtmw_swim.py
"""Fine-tune RTMW-x (whole-body pose, 133 points) on SwimXYZ swimmers: body and feet.

Part A reuses the frames already saved in the YOLO training shards (Drive) and only downloads
SwimXYZ's labels zip (about 1.7 GB), so the 79 GB of videos are not downloaded again.
Image names in the shards record the video and frame they came from: <view>__<swimmer>_<k>_f<frame>.jpg,
where k is the video's position in its shard's sorted video list.
"""
import io
import json
import re
import time
import zipfile
from pathlib import Path

import numpy as np
import requests

LABEL_ZIP = "https://zenodo.org/api/records/8399376/files/Freestyle_labels.zip/content"
PART2_ZIP = "https://zenodo.org/api/records/8402031/files/Freestyle_part2.zip/content"  # side views
PART1_RECORD = 8402009  # Aerial and Front views
PART1_VIEWS = {"Front", "Aerial"}
SRC_W, SRC_H = 1920, 1080  # SwimXYZ video size; labels are in these pixels
Y_SHIFT = 6  # SwimXYZ y is measured from the bottom (same fix as the YOLO notebook)

# OpenPose BODY_25 order: 0 nose, 1 neck, 2-4 right arm, 5-7 left arm, 8 mid-hip, 9-11 right leg,
# 12-14 left leg, 15/16 right/left eye, 17/18 right/left ear, 19-21 left big toe, small toe, heel,
# 22-24 right big toe, small toe, heel.
BODY25_TO_COCO17 = [0, 16, 15, 18, 17, 5, 2, 6, 3, 7, 4, 12, 9, 13, 10, 14, 11]
BODY25_TO_FEET = [19, 20, 21, 22, 23, 24]  # COCO-WholeBody points 17-22 use this order
COCO18_TO_COCO17 = [0, 15, 14, 17, 16, 5, 2, 6, 3, 7, 4, 11, 8, 12, 9, 13, 10]  # the YOLO notebook's mapping
IMAGE_NAME = re.compile(r"^(?P<shard>.+)_(?P<k>\d{3})_f(?P<frame>\d{3})(?P<zoom>_zoom)?\.jpg$")


# ---------- reading single files out of the big remote zips (same as the YOLO notebook) ----------

def request(session, method, url, **kw):
    """One HTTP request that waits out Zenodo's rate limit (HTTP 429) and retries network errors."""
    errors = 0
    for attempt in range(30):
        try:
            r = session.request(method, url, timeout=120, **kw)
            if r.status_code == 429:  # Zenodo rate limit: wait at least 30 s, longer each time
                wait = max(int(float(r.headers.get("Retry-After") or 0)), min(30 * (attempt + 1), 300))
                print(f"  Zenodo says too many requests, waiting {wait}s")
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r
        except requests.RequestException:
            errors += 1
            if errors == 8:
                raise
            time.sleep(2 ** errors)
    raise RuntimeError(f"Gave up on {url}: Zenodo kept saying too many requests. Wait an hour and run the cell again")


def download(url, path):
    """Download a file in one go (Zenodo allows few requests per minute, so this beats many small reads).
    Skipped if the file is already there; carries on from where it stopped if the download was cut off."""
    path = Path(path)
    if path.exists():
        return path
    part = path.with_name(path.name + ".part")
    session = requests.Session()
    for _ in range(20):
        done = part.stat().st_size if part.exists() else 0
        r = request(session, "get", url, stream=True, headers={"Range": f"bytes={done}-"} if done else {})
        if done and r.status_code != 206:  # the server ignored the resume request: start again
            done = 0
        total = done + int(r.headers.get("Content-Length", 0))
        t0, start, last = time.time(), done, done
        try:
            with open(part, "ab" if done else "wb") as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
                    done += len(chunk)
                    if done - last >= 200 << 20:
                        last = done
                        print(f"  {done / 1e9:.1f} of {total / 1e9:.1f} GB ({(done - start) / 1e6 / (time.time() - t0 + 1e-9):.0f} MB/s)")
        except requests.RequestException as e:
            print("  download cut off, carrying on:", e)
            continue
        if done >= total:
            part.rename(path)
            print(f"Downloaded {path.name} ({done / 1e9:.2f} GB)")
            return path
    raise RuntimeError(f"Could not download {url}")


class HttpFile(io.RawIOBase):
    """A seekable file over HTTP Range requests, so zipfile can pull single members."""

    def __init__(self, url):
        self.session = requests.Session()
        r = self._request("head", url, allow_redirects=True)
        self.url = r.url
        self.size = int(r.headers["Content-Length"])
        self.pos = 0

    def _request(self, method, url, **kw):
        return request(self.session, method, url, **kw)

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = {0: offset, 1: self.pos + offset, 2: self.size + offset}[whence]
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        if n == 0 or self.pos >= self.size:
            return b""
        end = min(self.pos + n, self.size) - 1
        r = self._request("get", self.url, headers={"Range": f"bytes={self.pos}-{end}"})
        if r.status_code != 206:
            raise RuntimeError("Server does not support Range requests")
        self.pos += len(r.content)
        return r.content

    def readinto(self, b):
        data = self.read(len(b))
        b[: len(data)] = data
        return len(data)


def open_remote_zip(url, buffer=1 << 22):
    return zipfile.ZipFile(io.BufferedReader(HttpFile(url), buffer))


def find_part1_url():
    """Download link of the zip in the "Freestyle videos (part 1)" Zenodo record."""
    api = f"https://zenodo.org/api/records/{PART1_RECORD}"
    try:
        zips = [f["key"] for f in requests.get(api, timeout=60).json()["files"] if f["key"].endswith(".zip")]
        if zips:
            return f"{api}/files/{zips[0]}/content"
    except Exception as e:
        print("Could not read the Zenodo record, guessing the file name:", e)
    return f"{api}/files/Freestyle_part1.zip/content"


def shard_name(view, swimmer):
    return f"{view}__{swimmer}".replace(",", "_")


def list_videos(part1_url):
    """{shard name: sorted video names}, the same lists (and order) the YOLO notebook used."""
    shards = {}
    for url in (part1_url, PART2_ZIP):
        for name in open_remote_zip(url).namelist():
            view = name.split("/")[0]
            if name.endswith(".webm") and (view in PART1_VIEWS) == (url == part1_url):
                shards.setdefault(shard_name(view, name.split("/")[1]), []).append(name)
    return {k: sorted(v) for k, v in shards.items()}


# ---------- labels ----------

def label_path(video, folder):
    return "Freestyle/" + video[: -len(".webm")] + f"/{folder}/2D_cam.txt"


def read_points(text, n_points):
    """Every frame of a SwimXYZ 2D_cam.txt as (frames, n_points, 2) pixels, y measured from the top."""
    rows = []
    for line in text.splitlines()[1:]:  # first line is the column names
        if not line.strip():
            continue
        v = [float(x.replace(",", ".")) for x in line.strip(";").split(";")]
        rows.append([(v[i * 3], SRC_H - v[i * 3 + 1] + Y_SHIFT) for i in range(n_points)])
    return np.array(rows, dtype=np.float64)


def label_folders(labels_zip, video):
    """{label folder: points per frame} for every 2D_cam.txt of one video."""
    prefix = "Freestyle/" + video[: -len(".webm")] + "/"
    found = {}
    for name in labels_zip.namelist():
        if name.startswith(prefix) and name.endswith("/2D_cam.txt"):
            text = labels_zip.read(name).decode()
            first = next(line for line in text.splitlines()[1:] if line.strip())
            found[name[len(prefix): -len("/2D_cam.txt")]] = len(first.strip(";").split(";")) / 3
    return found


def body25_order_check(points18, points25):
    """Which 25-point index sits on each of the 18 COCO joints (median over frames), and whether that
    matches OpenPose's BODY_25 order, which this notebook assumes."""
    expected = [0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]  # COCO-18 joint -> BODY_25 index
    dist = np.median(np.hypot(*(points18[:, :, None, :] - points25[:, None, :, :]).transpose(3, 0, 1, 2)), axis=0)
    found = [int(i) for i in dist.argmin(1)]
    return found == expected, found, float(np.median(dist.min(1)))


def inside(points):
    return (points[:, 0] >= 0) & (points[:, 0] < SRC_W) & (points[:, 1] >= 0) & (points[:, 1] < SRC_H)


def wholebody_annotation(points25, image_id, ann_id, img_w, img_h):
    """One COCO-WholeBody annotation: body and feet from SwimXYZ, face and hands left unlabeled (0)."""
    sx, sy = img_w / SRC_W, img_h / SRC_H

    def flat(points):
        out = []
        for (x, y), ok in zip(points, inside(points)):
            out += [round(x * sx, 2), round(y * sy, 2), 2] if ok else [0, 0, 0]
        return out

    body, feet = points25[BODY25_TO_COCO17], points25[BODY25_TO_FEET]
    seen = np.concatenate([body, feet])[inside(np.concatenate([body, feet]))]
    x1, y1 = seen.min(0)
    x2, y2 = seen.max(0)
    pad = 0.08 * max(x2 - x1, y2 - y1)
    x1, y1 = max(x1 - pad, 0) * sx, max(y1 - pad, 0) * sy
    x2, y2 = min(x2 + pad, SRC_W - 1) * sx, min(y2 + pad, SRC_H - 1) * sy
    return dict(
        id=ann_id, image_id=image_id, category_id=1, iscrowd=0, segmentation=[],
        bbox=[round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2)],
        area=round((x2 - x1) * (y2 - y1), 2), num_keypoints=int(len(seen)),
        keypoints=flat(body), foot_kpts=flat(feet),
        face_kpts=[0] * 68 * 3, lefthand_kpts=[0] * 21 * 3, righthand_kpts=[0] * 21 * 3,
    )


def yolo_points(label_line):
    """17 (x, y, visible) points from a YOLO pose label line (normalized)."""
    v = [float(x) for x in label_line.split()]
    return np.array(v[5:5 + 17 * 3]).reshape(17, 3)


# ---------- building the COCO-WholeBody files ----------

def shard_images(shard_zip):
    """Full-frame images in one shard zip: [(split, file name, k, frame)]. Zoomed crops are skipped:
    RTMW crops each swimmer from the frame itself, so it doesn't need them."""
    out = []
    with zipfile.ZipFile(shard_zip) as z:
        for name in z.namelist():
            parts = name.split("/")
            if len(parts) != 3 or parts[0] != "images":
                continue
            m = IMAGE_NAME.match(parts[2])
            if m and not m["zoom"]:
                out.append((parts[1], parts[2], int(m["k"]), int(m["frame"])))
    return out


def build(shards_dir, videos, folder25, out_dir, labels_zip, limit=None):
    """Write out_dir/train.json and val.json (COCO-WholeBody format) for every full frame in the shards,
    with labels from the downloaded SwimXYZ labels zip.

    Also compares the new body points with the YOLO labels saved next to each image, as a check
    that the 25-point labels line up. Returns that check's pixel differences.
    """
    from PIL import Image

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    coco = {split: dict(images=[], annotations=[], categories=[dict(id=1, name="person")])
            for split in ("train", "val")}
    diffs, feet_check, t0 = [], [], time.time()
    shard_zips = sorted(Path(shards_dir).glob("*.zip"))[:limit]
    labels = zipfile.ZipFile(labels_zip)
    for n_shard, shard_zip in enumerate(shard_zips, 1):
        shard = shard_zip.stem
        images = shard_images(shard_zip)
        by_video = {}
        for split, name, k, frame in images:
            by_video.setdefault(k, []).append((split, name, frame))

        with zipfile.ZipFile(shard_zip) as z:
            for k in sorted(by_video):
                points = read_points(labels.read(label_path(videos[shard][k], folder25)).decode(), 25)
                for split, name, frame in by_video[k]:
                    p25 = points[frame]
                    with z.open(f"images/{split}/{name}") as f:
                        img_w, img_h = Image.open(f).size
                    image_id = len(coco["train"]["images"]) + len(coco["val"]["images"]) + 1
                    coco[split]["images"].append(dict(id=image_id, file_name=f"{split}/{name}",
                                                      width=img_w, height=img_h))
                    coco[split]["annotations"].append(wholebody_annotation(p25, image_id, image_id, img_w, img_h))
                    # check: new body points vs the YOLO labels made from the 18-point files
                    old = yolo_points(z.read(f"labels/{split}/{name[:-4]}.txt").decode())
                    new = p25[BODY25_TO_COCO17] / [SRC_W, SRC_H]
                    seen = old[:, 2] > 0
                    diffs += list(np.hypot(*((new[seen] - old[seen, :2]) * [SRC_W, SRC_H]).T))
                    feet_check.append(_feet_side_check(p25))
        n = sum(len(c["images"]) for c in coco.values())
        print(f"{shard}: {len(images)} frames ({n_shard}/{len(shard_zips)} shards, {n} frames, {time.time() - t0:.0f}s)")
    labels.close()
    for split, data in coco.items():
        (out_dir / f"{split}.json").write_text(json.dumps(data))
        print(split, len(data["images"]), "frames ->", out_dir / f"{split}.json")
    return np.array(diffs), np.array(feet_check)


def _feet_side_check(p25):
    """True when each big toe is nearer its own ankle than the other ankle (catches a left/right mix-up)."""
    l_ankle, r_ankle, l_toe, r_toe = p25[14], p25[11], p25[19], p25[22]
    return (np.hypot(*(l_toe - l_ankle)) < np.hypot(*(l_toe - r_ankle))) and \
           (np.hypot(*(r_toe - r_ankle)) < np.hypot(*(r_toe - l_ankle)))


# ---------- training config ----------

def make_config(base_config, images_dir, ann_dir, work_dir, load_from, epochs, batch_size, lr, workers):
    """RTMW-x 384x288 settings from MMPose, pointed at our swim data, sized for one Colab T4."""
    from mmengine.config import Config

    cfg = Config.fromfile(base_config)
    cfg.model.backbone.init_cfg = None  # the whole model comes from load_from
    cfg.model.neck.norm_cfg = dict(type="BN")  # SyncBN is for multi-GPU training
    cfg.load_from = load_from
    cfg.work_dir = str(work_dir)
    cfg.train_cfg = dict(by_epoch=True, max_epochs=epochs, val_interval=5)
    cfg.optim_wrapper = dict(
        type="AmpOptimWrapper",  # mixed precision: faster on a T4 and uses less memory
        optimizer=dict(type="AdamW", lr=lr, weight_decay=0.05),
        clip_grad=dict(max_norm=35, norm_type=2),
        paramwise_cfg=dict(norm_decay_mult=0, bias_decay_mult=0, bypass_duplicate=True))
    cfg.param_scheduler = [
        dict(type="LinearLR", start_factor=0.01, by_epoch=False, begin=0, end=300),
        dict(type="CosineAnnealingLR", eta_min=lr * 0.05, begin=0, end=epochs, T_max=epochs,
             by_epoch=True, convert_to_iter_based=True),
    ]
    cfg.auto_scale_lr = dict(enable=False, base_batch_size=batch_size)

    # augmentations as in MMPose's config, with albumentations 2.x argument names
    train_pipeline = [
        dict(type="LoadImage"),
        dict(type="GetBBoxCenterScale"),
        dict(type="RandomFlip", direction="horizontal"),
        dict(type="RandomHalfBody"),
        dict(type="RandomBBoxTransform", scale_factor=[0.5, 1.5], rotate_factor=90),
        dict(type="TopdownAffine", input_size=cfg.codec["input_size"]),
        dict(type="PhotometricDistortion"),
        dict(type="Albumentation", transforms=[
            dict(type="Blur", p=0.1),
            dict(type="MedianBlur", p=0.1),
            dict(type="CoarseDropout", num_holes_range=(1, 1), hole_height_range=(0.2, 0.4),
                 hole_width_range=(0.2, 0.4), p=0.5),
        ]),
        dict(type="GenerateTarget", encoder=cfg.codec, use_dataset_keypoint_weights=True),
        dict(type="PackPoseInputs"),
    ]

    def dataset(split, pipeline, test_mode):
        return dict(type="CocoWholeBodyDataset", data_mode="topdown", ann_file=str(Path(ann_dir) / f"{split}.json"),
                    data_prefix=dict(img=str(images_dir) + "/"), pipeline=pipeline, test_mode=test_mode)

    cfg.train_dataloader = dict(
        batch_size=batch_size, num_workers=workers, persistent_workers=workers > 0, pin_memory=True,
        sampler=dict(type="DefaultSampler", shuffle=True), dataset=dataset("train", train_pipeline, False))
    cfg.val_dataloader = dict(
        batch_size=batch_size, num_workers=workers, persistent_workers=workers > 0, drop_last=False,
        sampler=dict(type="DefaultSampler", shuffle=False, round_up=False),
        dataset=dataset("val", cfg.val_pipeline, True))
    cfg.test_dataloader = cfg.val_dataloader
    # share of labeled points within 5% of the swimmer's box size (hands and face have no labels, so
    # this is body and feet)
    cfg.val_evaluator = dict(type="PCKAccuracy", thr=0.05, norm_item=["bbox"])
    cfg.test_evaluator = cfg.val_evaluator
    cfg.default_hooks.checkpoint = dict(type="CheckpointHook", interval=1, max_keep_ckpts=2,
                                        save_best="PCK", rule="greater")
    cfg.default_hooks.logger = dict(type="LoggerHook", interval=50)
    cfg.custom_hooks = [dict(type="EMAHook", ema_type="ExpMomentumEMA", momentum=0.0002,
                             update_buffers=True, priority=49)]
    return cfg


# ---------- checking and exporting ----------

def evaluate(model, ann_file, images_dir, limit=None):
    """PCK@0.05 (share of points within 5% of the box size) for body and feet, using the labeled box."""
    from mmpose.apis import inference_topdown

    data = json.loads(Path(ann_file).read_text())
    anns = {a["image_id"]: a for a in data["annotations"]}
    hits = {"body": [], "feet": []}
    for img in data["images"][:limit]:
        a = anns[img["id"]]
        x, y, w, h = a["bbox"]
        result = inference_topdown(model, str(Path(images_dir) / img["file_name"]),
                                   bboxes=np.array([[x, y, x + w, y + h]]), bbox_format="xyxy")[0]
        pred = result.pred_instances.keypoints[0]
        for part, key, idx in (("body", "keypoints", range(0, 17)), ("feet", "foot_kpts", range(17, 23))):
            gt = np.array(a[key]).reshape(-1, 3)
            seen = gt[:, 2] > 0
            dist = np.hypot(*(pred[list(idx)][seen] - gt[seen, :2]).T)
            hits[part] += list(dist < 0.05 * max(w, h))
    return {part: float(np.mean(v)) if v else float("nan") for part, v in hits.items()}


def export_onnx(model, path, input_size=(288, 384)):
    """ONNX file with the same inputs and outputs as MMPose's official RTMW downloads, so rtmlib
    (and the comparison notebook's Part 4) can run it."""
    import torch

    class SimCC(torch.nn.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m

        def forward(self, x):
            return self.m.head(self.m.extract_feat(x))

    w, h = input_size
    wrapper = SimCC(model).eval().cpu()
    torch.onnx.export(wrapper, torch.zeros(1, 3, h, w), str(path), input_names=["input"],
                      output_names=["simcc_x", "simcc_y"], opset_version=17, dynamo=False)
    return path

# Part A: labels with feet (CPU runtime is fine)

### A1. Find SwimXYZ's 25-point labels and check their order

This downloads SwimXYZ's labels zip (about 1.7 GB, a few minutes) to the Colab machine, once per session.
Then it looks at one video's label files and checks that the 25-point file follows OpenPose's BODY_25 order
(body, then 3 points per foot), by matching it against the 18-point labels the YOLO model was trained on.
If it says the order doesn't match, stop here and tell Claude what it printed.

In [ ]:
import sys, importlib
sys.path.insert(0, '/content')
import rtmw_swim as rs
importlib.reload(rs)

import json, zipfile
VIDEO_LIST = OUT / 'videos.json'   # saved so later sessions don't need to ask Zenodo again
if VIDEO_LIST.exists():
    videos = json.loads(VIDEO_LIST.read_text())
else:
    videos = rs.list_videos(rs.find_part1_url())
    VIDEO_LIST.write_text(json.dumps(videos))
print(len(videos), 'shards,', sum(len(v) for v in videos.values()), 'videos')
LABELS = rs.download(rs.LABEL_ZIP, Path('/content/Freestyle_labels.zip'))
labels = zipfile.ZipFile(LABELS)
first = videos[sorted(videos)[0]][0]
folders = rs.label_folders(labels, first)
print('Label folders for', first, '->', folders)
FOLDER25 = next((f for f, n in folders.items() if n == 25), None)
assert FOLDER25, 'No 25-point label file found. Tell Claude what the line above printed.'
p18 = rs.read_points(labels.read(rs.label_path(first, 'COCO')).decode(), 18)
p25 = rs.read_points(labels.read(rs.label_path(first, FOLDER25)).decode(), 25)
same, found, px = rs.body25_order_check(p18, p25)
print(f'25-point folder: {FOLDER25}. Body joints line up with the 18-point labels within {px:.1f} px.')
print('Order matches BODY_25: YES' if same else f'Order does NOT match BODY_25. Tell Claude: {found}')

### A2. Try one shard and look at the labels

Builds the labels for one shard into a test folder, checks them, and draws a few.
Blue = swimmer's left, orange = right; big dots are the feet (toes and heel).

In [ ]:
import json, zipfile
import cv2, numpy as np, matplotlib.pyplot as plt

test_dir = Path('/content/rtmw_test')
diffs, feet_ok = rs.build(SHARDS, videos, FOLDER25, test_dir, LABELS, limit=1)
print(f'Body points vs the YOLO labels: median {np.median(diffs):.1f} px, 95% within {np.percentile(diffs, 95):.1f} px')
print(f'Big toes nearer their own ankle: {feet_ok.mean():.0%} of frames')
print('Looks right' if np.median(diffs) < 15 and feet_ok.mean() > 0.9 else 'Something is off: tell Claude these numbers')

images, anns = [], []
for split in ('train', 'val'):
    d = json.loads((test_dir / f'{split}.json').read_text())
    images, anns = images + d['images'], anns + d['annotations']
LEFT = [1, 3, 5, 7, 9, 11, 13, 15, 17, 18, 19]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
with zipfile.ZipFile(sorted(SHARDS.glob('*.zip'))[0]) as z:
    for ax, i in zip(axes, np.linspace(0, len(images) - 1, 4).astype(int)):
        img_info, ann = images[i], anns[i]
        img = cv2.imdecode(np.frombuffer(z.read('images/' + img_info['file_name']), np.uint8), cv2.IMREAD_COLOR)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        pts = np.array(ann['keypoints'] + ann['foot_kpts']).reshape(-1, 3)
        for i, (x, y, v) in enumerate(pts):
            if v > 0:
                ax.plot(x, y, 'o', color='tab:blue' if i in LEFT else 'tab:orange', markersize=7 if i >= 17 else 3)
        ax.axis('off')
plt.show()

### A3. Build the labels for all 20 shards

Reads each video's label file from the labels zip that A1 downloaded, so it's much quicker than building the shards was.
Saves `train.json` and `val.json` to `MyDrive/aquatrack_swim_full/rtmw/annotations/`.

In [ ]:
diffs, feet_ok = rs.build(SHARDS, videos, FOLDER25, ANN, LABELS)
print(f'Body points vs the YOLO labels: median {np.median(diffs):.1f} px; big toes on the right side {feet_ok.mean():.0%}')

# Part B: train (switch to a GPU first)

**Runtime > Change runtime type**: with Colab Pro pick **A100** (fastest) or **L4**, otherwise **T4**. Then run **0. Setup**, the helper cell, and the cells below.
If Colab disconnects, do the same: B2 notices the saved checkpoint and carries on.

### B1. Copy the frames onto the fast local disk

In [ ]:
import zipfile
import torch
if not torch.cuda.is_available():
    print('No GPU right now: copying works, but switch to a GPU before B2 (Runtime > Change runtime type).')
if not (DATASET / 'images').exists():
    for z in sorted(SHARDS.glob('*.zip')):
        with zipfile.ZipFile(z) as zz:
            zz.extractall(DATASET, [n for n in zz.namelist() if n.startswith('images/') and '_zoom' not in n])
for split in ('train', 'val'):
    print(split, len(list((DATASET / 'images' / split).glob('*.jpg'))), 'frames')

### B2. Train (or carry on)

Starts from the official RTMW-x and saves a checkpoint to Drive after every epoch.
Every 5 epochs it scores the held-out swimmer (PCK: share of body and foot points within 5% of the swimmer's size).
It picks the batch size from the GPU's memory. If you see `CUDA out of memory`, set `BATCH` to half of what it printed.

In [ ]:
import sys, importlib
sys.path.insert(0, '/content')
import rtmw_swim as rs
import torch
from mmengine.runner import Runner
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type (A100, L4 or T4), then run Setup, the helper cell and B1 again'

import os
EPOCHS = 30
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
BATCH = 64 if GPU_GB > 22 else 32   # A100 / L4: 64, T4: 32. Lower it if you see `CUDA out of memory`
LR = 4e-5 * BATCH / 32               # small: RTMW-x is already well trained, we only adjust it to swimmers
WORKERS = min(8, os.cpu_count())     # bigger Colab Pro machines have more CPUs to prepare images
print(f'{torch.cuda.get_device_name(0) if GPU_GB else "no GPU"}: batch {BATCH}, lr {LR:.0e}, {WORKERS} workers')

cfg = rs.make_config(BASE_CONFIG, DATASET / 'images', ANN, RUN, load_from=RTMW_X,
                     epochs=EPOCHS, batch_size=BATCH, lr=LR, workers=WORKERS)
if (RUN / 'last_checkpoint').exists():
    cfg.resume, cfg.load_from = True, None   # carry on from the last saved epoch
    print('Carrying on from', (RUN / 'last_checkpoint').read_text().strip())
Runner.from_cfg(cfg).train()

# Part C: check and export (any runtime; a GPU is faster)

### C1. Export ONNX files

Saves `rtmw_x_stock.onnx` (the official RTMW-x) and, once training has run, `rtmw_x_swim.onnx`, to
`MyDrive/aquatrack_swim_full/rtmw/`. These are what rtmlib, Part 4 of the comparison notebook and later the app can run.

In [ ]:
import sys, importlib
sys.path.insert(0, '/content')
import rtmw_swim as rs
import torch
from mmpose.apis import init_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = rs.make_config(BASE_CONFIG, DATASET / 'images', ANN, RUN, load_from=None, epochs=1, batch_size=1, lr=0, workers=0)
trained = sorted(RUN.glob('best_PCK_*.pth')) or sorted(RUN.glob('epoch_*.pth'), key=lambda p: int(p.stem.split('_')[1]))
MODELS = {'stock': RTMW_X}
if trained:
    MODELS['swim'] = str(trained[-1])
for name, ckpt in MODELS.items():
    model = init_model(cfg, ckpt, device='cpu')
    path = rs.export_onnx(model, OUT / f'rtmw_x_{name}.onnx')
    print(f'{name}: {ckpt} -> {path}')
print()
print('To try them in Part 4 of the comparison notebook, add these lines to WHOLEBODY in cell 8:')
for name in MODELS:
    print(f"    'RTMW-x {name}': ('rtmw', '{OUT / f'rtmw_x_{name}.onnx'}', (288, 384)),")

### C2. Scores on the held-out swimmer

PCK = share of points within 5% of the swimmer's size, for the body (17) and feet (6), on the swimmer kept
out of training. These frames are computer-generated, so the real test is your own clips (Part 4 of the
comparison notebook) and real labeled frames.
Needs the frames on the local disk: run B1 first if this is a new session.

In [ ]:
LIMIT = None   # e.g. 200 for a quick look on a CPU runtime
for name, ckpt in MODELS.items():
    model = init_model(cfg, ckpt, device=device)
    scores = rs.evaluate(model, ANN / 'val.json', DATASET / 'images', limit=LIMIT)
    print(f"{name:6s}  body {scores['body']:.1%}   feet {scores['feet']:.1%}")